# Baseline y Modelos (Fase C.5)
Orquestación de entrenamiento, validación cruzada, y elección de umbral.

In [1]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from src.data import load_train
from src.models import BASELINES, MODELS, make_model
from src.evaluation import cross_validate_oof, summarize_folds, paired_differences, paired_bootstrap_diff, choose_threshold, segment_errors
from src.train import train_and_save

X, y, df = load_train()
print(f'Train shape: {X.shape}, Positives: {y.sum()} ({y.mean():.2%})')

Train shape: (3539, 31), Positives: 1137 (32.13%)


### 1. Baselines

In [2]:
results_oof = {}
fold_tables = {}

for name in BASELINES:
    print(f'Evaluando {name}...')
    df_folds, oof = cross_validate_oof(make_model(name), X, y)
    fold_tables[name] = df_folds
    results_oof[name] = oof

display(summarize_folds(fold_tables).loc[BASELINES])

Evaluando dummy_prior...


Evaluando dummy_todos_dropout...
Evaluando regla_aprobadas0...


,pr_auc_media,pr_auc_sd,roc_auc_media,roc_auc_sd,brier_media,brier_sd,f1@0.5_media,f1@0.5_sd
modelo,,,,,,,,
dummy_prior,0.321277,0.000653,0.500000,0.000000,0.218058,0.000233,0.000000,0.000000
dummy_todos_dropout,0.321277,0.000653,0.500000,0.000000,0.678723,0.000653,0.486313,0.000748
regla_aprobadas0,0.504426,0.021182,0.673022,0.014553,0.229443,0.010911,0.528095,0.027336


### 2. Modelos (LogReg, Random Forest, HistGradientBoosting)

In [3]:
for name in MODELS:
    print(f'Evaluando {name}...')
    df_folds, oof = cross_validate_oof(make_model(name), X, y)
    fold_tables[name] = df_folds
    results_oof[name] = oof

display(summarize_folds(fold_tables).loc[MODELS])

print('\nDiferencias pareadas vs Regla de Aprobadas = 0:')
display(paired_differences(fold_tables, ref='regla_aprobadas0'))

Evaluando logreg...


Evaluando random_forest...


Evaluando hist_gb...


,pr_auc_media,pr_auc_sd,roc_auc_media,roc_auc_sd,brier_media,brier_sd,f1@0.5_media,f1@0.5_sd
modelo,,,,,,,,
logreg,0.860749,0.018567,0.903544,0.015798,0.104666,0.008213,0.761822,0.025268
random_forest,0.844654,0.019757,0.893621,0.013813,0.113329,0.006213,0.736113,0.018242
hist_gb,0.855360,0.018097,0.899176,0.014396,0.107591,0.008221,0.752793,0.023095



Diferencias pareadas vs Regla de Aprobadas = 0:


,ref,dif_media,dif_sd,folds_modelo_gana,n_folds,diferencia_real(|media|>=1sd)
modelo,,,,,,
dummy_prior,regla_aprobadas0,-0.183149,0.021113,0,25,True
dummy_todos_dropout,regla_aprobadas0,-0.183149,0.021113,0,25,True
logreg,regla_aprobadas0,0.356324,0.020922,25,25,True
random_forest,regla_aprobadas0,0.340228,0.019001,25,25,True
hist_gb,regla_aprobadas0,0.350934,0.019567,25,25,True


### 3. Diferencias Pareadas entre Modelos (OOF Bootstrap)

In [4]:
diff, lo, hi = paired_bootstrap_diff(y, results_oof['hist_gb'], results_oof['logreg'])
print(f'HGB vs LogReg (PR-AUC): Diff={diff:.4f} 95% CI [{lo:.4f}, {hi:.4f}]')

diff_rf, lo_rf, hi_rf = paired_bootstrap_diff(y, results_oof['random_forest'], results_oof['logreg'])
print(f'RF vs LogReg (PR-AUC): Diff={diff_rf:.4f} 95% CI [{lo_rf:.4f}, {hi_rf:.4f}]')

best_model = 'logreg' if lo <= 0 and lo_rf <= 0 else ('hist_gb' if diff > diff_rf else 'random_forest')
print(f'\nModelo seleccionado: {best_model}')

HGB vs LogReg (PR-AUC): Diff=-0.0008 95% CI [-0.0070, 0.0054]


RF vs LogReg (PR-AUC): Diff=-0.0168 95% CI [-0.0249, -0.0084]

Modelo seleccionado: logreg


### 4. Ablación

In [5]:
from src import config as C
for ablation_name, cols_to_drop, drop_flags in [
    ('Sin Financieras', C.FINANCIAL_COLS, False),
    ('Sin Aprobadas', C.APPROVED_SIGNAL_COLS, True)
]:
    print(f'\nAblación: {ablation_name}')
    df_folds_ab, oof_ab = cross_validate_oof(make_model(best_model, drop=cols_to_drop, add_flags=not drop_flags), X, y)
    diff_ab, lo_ab, hi_ab = paired_bootstrap_diff(y, results_oof[best_model], oof_ab)
    print(f'Impacto (Diff OOF PR-AUC original vs ablación): {diff_ab:.4f} 95% CI [{lo_ab:.4f}, {hi_ab:.4f}]')


Ablación: Sin Financieras


Impacto (Diff OOF PR-AUC original vs ablación): 0.0383 95% CI [0.0288, 0.0484]

Ablación: Sin Aprobadas


Impacto (Diff OOF PR-AUC original vs ablación): 0.1167 95% CI [0.1014, 0.1330]


### 5. Selección de Umbral en OOF (y Entrenamiento Final)

In [6]:
thr, prec, rec = choose_threshold(y, results_oof[best_model], min_recall=0.85)
print(f'Umbral seleccionado: {thr:.4f} (Recall OOF: {rec:.4f}, Prec OOF: {prec:.4f})')

print('\nEntrenando modelo final en todo train y guardando...')
pipe, meta = train_and_save(best_model, thr, summarize_folds({best_model: fold_tables[best_model]}).iloc[0].to_dict())
print(f"Guardado exitoso. Hash: {meta['modelo_sha256']}")

Umbral seleccionado: 0.2423 (Recall OOF: 0.8505, Prec OOF: 0.6739)

Entrenando modelo final en todo train y guardando...


Guardado exitoso. Hash: 6588bf536ebc1b2f4424994fee19a7ce1d9f6051744ec1708160be105f758cca


### 6. Análisis Preliminar de Errores por Segmento (sobre OOF)

In [7]:
for segment in C.SENSITIVE_COLS:
    err = segment_errors(df, y, results_oof[best_model], thr, segment)
    display(err[['segmento', 'grupo', 'n', 'FNR', 'FNR_IC95', 'FPR', 'FPR_IC95']])

,segmento,grupo,n,FNR,FNR_IC95,FPR,FPR_IC95
0,Gender,0,2304,0.201031,"[0.17, 0.24]",0.140534,"[0.12, 0.16]"
1,Gender,1,1235,0.095495,"[0.07, 0.12]",0.332353,"[0.30, 0.37]"


,segmento,grupo,n,FNR,FNR_IC95,FPR,FPR_IC95
0,Nacionality,1,3455,0.149146,"[0.13, 0.17]",0.195559,"[0.18, 0.21]"
1,Nacionality,2,1,NaN,"[nan, nan]",0.000000,"[0.00, 0.79]"
2,Nacionality,6,10,0.000000,"[0.00, 0.56]",0.142857,"[0.03, 0.51]"
3,Nacionality,11,3,NaN,"[nan, nan]",0.000000,"[0.00, 0.56]"
4,Nacionality,13,1,NaN,"[nan, nan]",0.000000,"[0.00, 0.79]"
5,Nacionality,14,1,NaN,"[nan, nan]",0.000000,"[0.00, 0.79]"
6,Nacionality,17,1,0.000000,"[0.00, 0.79]",NaN,"[nan, nan]"
7,Nacionality,21,2,0.000000,"[0.00, 0.79]",1.000000,"[0.21, 1.00]"
8,Nacionality,22,10,0.000000,"[0.00, 0.56]",0.285714,"[0.08, 0.64]"
9,Nacionality,24,3,NaN,"[nan, nan]",0.333333,"[0.06, 0.79]"


,segmento,grupo,n,FNR,FNR_IC95,FPR,FPR_IC95
0,Marital status,1,3144,0.165971,"[0.14, 0.19]",0.187557,"[0.17, 0.20]"
1,Marital status,2,296,0.058824,"[0.03, 0.11]",0.268750,"[0.21, 0.34]"
2,Marital status,3,4,0.000000,"[0.00, 0.79]",0.666667,"[0.21, 0.94]"
3,Marital status,4,69,0.066667,"[0.02, 0.21]",0.256410,"[0.15, 0.41]"
4,Marital status,5,22,0.111111,"[0.02, 0.44]",0.230769,"[0.08, 0.50]"
5,Marital status,6,4,0.000000,"[0.00, 0.56]",0.000000,"[0.00, 0.79]"


,segmento,grupo,n,FNR,FNR_IC95,FPR,FPR_IC95
0,Scholarship holder,0,2671,0.119186,"[0.10, 0.14]",0.256254,"[0.24, 0.28]"
1,Scholarship holder,1,868,0.447619,"[0.36, 0.54]",0.062910,"[0.05, 0.08]"


,segmento,grupo,n,FNR,FNR_IC95,FPR,FPR_IC95
0,Age at enrollment,17,5,NaN,"[nan, nan]",0.200000,"[0.04, 0.62]"
1,Age at enrollment,18,809,0.378882,"[0.31, 0.46]",0.121914,"[0.10, 0.15]"
2,Age at enrollment,19,726,0.225610,"[0.17, 0.30]",0.154804,"[0.13, 0.19]"
3,Age at enrollment,20,498,0.139130,"[0.09, 0.21]",0.174935,"[0.14, 0.22]"
4,Age at enrollment,21,273,0.133333,"[0.07, 0.23]",0.186869,"[0.14, 0.25]"
5,Age at enrollment,22,140,0.152174,"[0.08, 0.28]",0.297872,"[0.21, 0.40]"
6,Age at enrollment,23,83,0.156250,"[0.07, 0.32]",0.254902,"[0.16, 0.39]"
7,Age at enrollment,24,100,0.113636,"[0.05, 0.24]",0.214286,"[0.13, 0.34]"
8,Age at enrollment,25,75,0.047619,"[0.01, 0.16]",0.303030,"[0.17, 0.47]"
9,Age at enrollment,26,77,0.068182,"[0.02, 0.18]",0.454545,"[0.30, 0.62]"
